# Linear Regression Implementation

April 15th 2026

We made tensors ML ready in `Make_tensors.ipynb`.

In this notebook, we load the saved tensor bundles, configure dataset splits, and build PyTorch `DataLoader`s for downstream linear regression experiments.

Supported split modes
- `drug_blind`: hold out entire drugs across all cell lines
- `tumor_blind`: hold out entire cell lines (this notebook uses cell-line-blind behavior)
- `mixed`: hold out specific cell line + drug + concentration combinations while keeping every drug and cell line in train


In [1]:
from pathlib import Path

SPLIT_MODE = "mixed"  # one of {"drug_blind", "tumor_blind", "mixed"}
SPLIT_FRACTIONS = {"train": 0.8, "val": 0.1, "test": 0.1}
BATCH_SIZE = 512
RANDOM_SEED = 42
NUM_WORKERS = 0

TENSOR_ARTIFACTS_DIR = Path("data/Tahoe100M_tensor_artifacts")

print(
    {
        "SPLIT_MODE": SPLIT_MODE,
        "SPLIT_FRACTIONS": SPLIT_FRACTIONS,
        "BATCH_SIZE": BATCH_SIZE,
        "RANDOM_SEED": RANDOM_SEED,
        "NUM_WORKERS": NUM_WORKERS,
        "tumor_blind_behavior": "cell_line_blind",
    }
)


{'SPLIT_MODE': 'mixed', 'SPLIT_FRACTIONS': {'train': 0.8, 'val': 0.1, 'test': 0.1}, 'BATCH_SIZE': 512, 'RANDOM_SEED': 42, 'NUM_WORKERS': 0, 'tumor_blind_behavior': 'cell_line_blind'}


## Imports and Helpers


In [2]:
import math

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.ipc as ipc
import torch
from IPython.display import display
from torch.utils.data import DataLoader, Dataset

SPLIT_NAMES = ("train", "val", "test")
SUPPORTED_SPLIT_MODES = {"drug_blind", "tumor_blind", "mixed"}


def resolve_project_path(relative_path):
    path = Path(relative_path)
    if path.is_absolute():
        return path

    for base_path in [Path.cwd(), *Path.cwd().parents]:
        candidate_path = base_path / path
        if candidate_path.exists():
            return candidate_path

    raise FileNotFoundError(f"Could not find {path} from {Path.cwd()}")


def load_arrow_table(arrow_path):
    with pa.memory_map(str(arrow_path), "r") as source:
        try:
            return ipc.open_file(source).read_all()
        except pa.ArrowInvalid:
            source.seek(0)
            return ipc.open_stream(source).read_all()


def load_cached_cell_line_metadata():
    arrow_candidates = sorted(
        Path.home().glob(
            ".cache/huggingface/datasets/vevotx___tahoe-100_m/cell_line_metadata/0.0.0/*/tahoe-100_m-train.arrow"
        )
    )
    if not arrow_candidates:
        raise FileNotFoundError(
            "Could not find cached Tahoe cell_line_metadata Arrow file in the Hugging Face cache."
        )

    arrow_path = arrow_candidates[-1]
    raw_df = load_arrow_table(arrow_path).to_pandas().loc[:, ["Cell_ID_Cellosaur", "cell_name", "Organ"]].copy()
    raw_df.columns = ["cell_line", "cell_name", "organ"]

    for column in raw_df.columns:
        raw_df[column] = raw_df[column].astype(str).str.strip()

    consistency_df = raw_df.groupby("cell_line", dropna=False).agg(
        cell_name_nunique=("cell_name", lambda s: s.nunique(dropna=False)),
        organ_nunique=("organ", lambda s: s.nunique(dropna=False)),
    )
    inconsistent_df = consistency_df.loc[
        (consistency_df["cell_name_nunique"] != 1)
        | (consistency_df["organ_nunique"] != 1)
    ]
    if not inconsistent_df.empty:
        raise ValueError(
            "Some Cellosaur IDs map to multiple cell_name/Organ values; cannot build a stable cell-line metadata table."
        )

    cell_line_metadata_df = (
        raw_df.groupby("cell_line", as_index=False, dropna=False)
        .first()
        .sort_values("cell_line", ignore_index=True)
    )
    return cell_line_metadata_df, arrow_path


def validate_split_config(split_mode, split_fractions):
    if split_mode not in SUPPORTED_SPLIT_MODES:
        raise ValueError(f"Unsupported SPLIT_MODE: {split_mode}")

    if set(split_fractions) != set(SPLIT_NAMES):
        raise ValueError(
            f"SPLIT_FRACTIONS must contain exactly {SPLIT_NAMES}; got {tuple(split_fractions)}"
        )

    if any(float(split_fractions[name]) <= 0 for name in SPLIT_NAMES):
        raise ValueError("All split fractions must be positive.")

    total_fraction = sum(float(split_fractions[name]) for name in SPLIT_NAMES)
    if not math.isclose(total_fraction, 1.0, abs_tol=1e-8):
        raise ValueError(f"Split fractions must sum to 1.0; got {total_fraction}")


def compute_target_row_counts(n_rows, split_fractions):
    val_rows = int(round(float(split_fractions["val"]) * n_rows))
    test_rows = int(round(float(split_fractions["test"]) * n_rows))
    train_rows = int(n_rows - val_rows - test_rows)

    if min(train_rows, val_rows, test_rows) <= 0:
        raise ValueError(
            f"Split fractions produced an empty split: train={train_rows}, val={val_rows}, test={test_rows}"
        )

    return {"train": train_rows, "val": val_rows, "test": test_rows}


def assign_group_blind_splits(frame, group_col, split_fractions, seed):
    target_counts = compute_target_row_counts(len(frame), split_fractions)
    group_sizes = frame.groupby(group_col, dropna=False).size().reset_index(name="row_count")

    rng = np.random.default_rng(seed)
    group_sizes["shuffle_order"] = rng.permutation(len(group_sizes))
    group_sizes = group_sizes.sort_values(
        ["row_count", "shuffle_order"],
        ascending=[False, True],
        kind="stable",
        ignore_index=True,
    )

    assigned_counts = {split_name: 0 for split_name in SPLIT_NAMES}
    group_to_split = {}

    for group_idx, row in group_sizes.iterrows():
        remaining_groups = len(group_sizes) - group_idx
        empty_splits = [split_name for split_name in SPLIT_NAMES if assigned_counts[split_name] == 0]
        if empty_splits and remaining_groups == len(empty_splits):
            candidate_splits = tuple(empty_splits)
        else:
            candidate_splits = SPLIT_NAMES

        group_size = int(row["row_count"])

        def assignment_score(split_name):
            projected = assigned_counts[split_name] + group_size
            target = target_counts[split_name]
            return (
                projected > target,
                abs(target - projected),
                assigned_counts[split_name] / max(target, 1),
                SPLIT_NAMES.index(split_name),
            )

        chosen_split = min(candidate_splits, key=assignment_score)
        group_to_split[row[group_col]] = chosen_split
        assigned_counts[chosen_split] += group_size

    split_series = frame[group_col].map(group_to_split)
    if split_series.isna().any():
        raise ValueError(f"Failed to assign every {group_col} to a split.")

    return split_series.astype(str)


def assign_mixed_split(frame, split_fractions, seed):
    target_counts = compute_target_row_counts(len(frame), split_fractions)
    split_assignments = np.full(len(frame), "train", dtype=object)
    remaining_drug_counts = frame["drug"].value_counts().to_dict()
    remaining_cell_line_counts = frame["cell_line"].value_counts().to_dict()
    drugs = frame["drug"].to_numpy()
    cell_lines = frame["cell_line"].to_numpy()

    rng = np.random.default_rng(seed)
    for split_name in ("val", "test"):
        target_rows = target_counts[split_name]
        assigned_rows = 0

        for row_idx in rng.permutation(len(frame)):
            if split_assignments[row_idx] != "train":
                continue

            drug = drugs[row_idx]
            cell_line = cell_lines[row_idx]
            if remaining_drug_counts[drug] <= 1 or remaining_cell_line_counts[cell_line] <= 1:
                continue

            split_assignments[row_idx] = split_name
            remaining_drug_counts[drug] -= 1
            remaining_cell_line_counts[cell_line] -= 1
            assigned_rows += 1

            if assigned_rows >= target_rows:
                break

        if assigned_rows < target_rows:
            raise ValueError(
                f"Could only assign {assigned_rows} rows to {split_name} while preserving train coverage for every drug and cell_line."
            )

    return pd.Series(split_assignments, index=frame.index, name="split")


def build_split_summary(split_frame, split_fractions):
    target_counts = compute_target_row_counts(len(split_frame), split_fractions)
    summary_rows = []

    for split_name in SPLIT_NAMES:
        subset = split_frame.loc[split_frame["split"] == split_name].copy()
        summary_rows.append(
            {
                "split": split_name,
                "target_rows": target_counts[split_name],
                "row_count": int(len(subset)),
                "requested_fraction": float(split_fractions[split_name]),
                "realized_fraction": float(len(subset) / len(split_frame)),
                "unique_drugs": int(subset["drug"].nunique()),
                "unique_cell_lines": int(subset["cell_line"].nunique()),
                "unique_organs": int(subset["organ"].nunique()),
            }
        )

    return pd.DataFrame(summary_rows)


def build_overlap_diagnostics(split_frame, split_mode):
    if split_mode == "drug_blind":
        entity_columns = ("drug", "condition_key", "cell_line")
    elif split_mode == "tumor_blind":
        entity_columns = ("cell_line", "condition_key", "drug")
    else:
        entity_columns = ("condition_key", "drug", "cell_line")

    split_sets = {
        split_name: {
            column: set(split_frame.loc[split_frame["split"] == split_name, column])
            for column in entity_columns
        }
        for split_name in SPLIT_NAMES
    }

    rows = []
    for column in entity_columns:
        for left_split, right_split in (("train", "val"), ("train", "test"), ("val", "test")):
            rows.append(
                {
                    "entity": column,
                    "pair": f"{left_split}/{right_split}",
                    "overlap_count": len(split_sets[left_split][column] & split_sets[right_split][column]),
                }
            )

    return pd.DataFrame(rows)


def validate_split_assignments(split_frame, split_mode):
    if split_frame["condition_key"].duplicated().any():
        raise ValueError("condition_key values must remain unique after splitting.")

    if set(split_frame["split"]) != set(SPLIT_NAMES):
        raise ValueError("Every split must contain at least one row.")

    if split_mode == "drug_blind":
        split_sets = {
            split_name: set(split_frame.loc[split_frame["split"] == split_name, "drug"])
            for split_name in SPLIT_NAMES
        }
        for left_split, right_split in (("train", "val"), ("train", "test"), ("val", "test")):
            if split_sets[left_split] & split_sets[right_split]:
                raise ValueError("drug_blind split leaked drugs across splits.")

    elif split_mode == "tumor_blind":
        split_sets = {
            split_name: set(split_frame.loc[split_frame["split"] == split_name, "cell_line"])
            for split_name in SPLIT_NAMES
        }
        for left_split, right_split in (("train", "val"), ("train", "test"), ("val", "test")):
            if split_sets[left_split] & split_sets[right_split]:
                raise ValueError("tumor_blind split leaked cell lines across splits.")

    else:
        condition_key_sets = {
            split_name: set(split_frame.loc[split_frame["split"] == split_name, "condition_key"])
            for split_name in SPLIT_NAMES
        }
        for left_split, right_split in (("train", "val"), ("train", "test"), ("val", "test")):
            if condition_key_sets[left_split] & condition_key_sets[right_split]:
                raise ValueError("mixed split leaked condition keys across splits.")

        train_frame = split_frame.loc[split_frame["split"] == "train"]
        if set(train_frame["drug"]) != set(split_frame["drug"]):
            raise ValueError("mixed split must retain every drug in train.")
        if set(train_frame["cell_line"]) != set(split_frame["cell_line"]):
            raise ValueError("mixed split must retain every cell line in train.")


class TreatmentExampleDataset(Dataset):
    def __init__(self, examples_df, dmso_bundle, treatment_bundle, fingerprint_bundle):
        self.examples_df = examples_df.reset_index(drop=True).copy()
        self.dmso_bundle = dmso_bundle
        self.treatment_bundle = treatment_bundle
        self.fingerprint_bundle = fingerprint_bundle

    def __len__(self):
        return len(self.examples_df)

    def __getitem__(self, idx):
        row = self.examples_df.iloc[idx]
        baseline_index = int(row["baseline_index"])
        fingerprint_index = int(row["fingerprint_index"])
        target_index = int(row["target_index"])

        return {
            "baseline_expression": self.dmso_bundle["expressions"][baseline_index],
            "drug_fingerprint": self.fingerprint_bundle["fingerprints"][fingerprint_index],
            "concentration": torch.tensor(float(row["concentration"]), dtype=torch.float32),
            "target_expression": self.treatment_bundle["expressions"][target_index],
            "condition_key": row["condition_key"],
            "cell_line": row["cell_line"],
            "cell_name": row["cell_name"],
            "organ": row["organ"],
            "drug": row["drug"],
            "concentration_unit": row["concentration_unit"],
        }


## Load Tensors and Supporting Metadata


In [3]:
validate_split_config(SPLIT_MODE, SPLIT_FRACTIONS)

tensor_artifacts_dir = resolve_project_path(TENSOR_ARTIFACTS_DIR)
dmso_bundle = torch.load(tensor_artifacts_dir / "dmso_baselines.pt", map_location="cpu")
treatment_bundle = torch.load(tensor_artifacts_dir / "treatment_expressions.pt", map_location="cpu")
fingerprint_bundle = torch.load(tensor_artifacts_dir / "morgan_fingerprints.pt", map_location="cpu")

if dmso_bundle["gene_ids"] != treatment_bundle["gene_ids"]:
    raise ValueError("DMSO and treatment bundles do not share the same gene IDs/order.")

cell_line_metadata_df, cell_line_metadata_arrow_path = load_cached_cell_line_metadata()
tensor_cell_lines = set(dmso_bundle["cell_lines"])
matched_cell_line_metadata_df = cell_line_metadata_df.loc[
    cell_line_metadata_df["cell_line"].isin(tensor_cell_lines)
].copy()
missing_cell_line_metadata = sorted(tensor_cell_lines - set(matched_cell_line_metadata_df["cell_line"]))
if missing_cell_line_metadata:
    raise ValueError(f"Missing cell-line metadata for: {missing_cell_line_metadata}")

if matched_cell_line_metadata_df["cell_line"].duplicated().any():
    raise ValueError("cell_line metadata must be unique after deduplication.")

examples_df = pd.DataFrame(
    {
        "condition_key": treatment_bundle["condition_keys"],
        "cell_line": treatment_bundle["cell_lines"],
        "file_name": treatment_bundle["file_names"],
        "drug": treatment_bundle["drug_names"],
        "concentration": treatment_bundle["concentrations"].cpu().numpy().astype(np.float32),
        "concentration_unit": treatment_bundle["concentration_units"],
        "target_index": np.arange(len(treatment_bundle["condition_keys"]), dtype=np.int64),
    }
)
examples_df["baseline_index"] = examples_df["cell_line"].map(dmso_bundle["cell_line_to_index"])
examples_df["fingerprint_index"] = examples_df["drug"].map(fingerprint_bundle["drug_to_index"])

if examples_df["condition_key"].duplicated().any():
    raise ValueError("condition_key values must be unique in the treatment bundle.")
if examples_df["baseline_index"].isna().any():
    raise ValueError("Some treatment rows do not resolve to a DMSO baseline index.")
if examples_df["fingerprint_index"].isna().any():
    raise ValueError("Some treatment rows do not resolve to a Morgan fingerprint index.")

examples_df = examples_df.merge(
    matched_cell_line_metadata_df,
    on="cell_line",
    how="left",
    validate="many_to_one",
)
if examples_df[["cell_name", "organ"]].isna().any().any():
    raise ValueError("Some treatment rows do not resolve to cell-line metadata.")

examples_df[["baseline_index", "fingerprint_index", "target_index"]] = examples_df[["baseline_index", "fingerprint_index", "target_index"]].astype(int)

tensor_summary_df = pd.DataFrame(
    [
        {"bundle": "dmso_baselines", "rows": int(dmso_bundle["expressions"].shape[0]), "cols": int(dmso_bundle["expressions"].shape[1])},
        {"bundle": "treatment_expressions", "rows": int(treatment_bundle["expressions"].shape[0]), "cols": int(treatment_bundle["expressions"].shape[1])},
        {"bundle": "morgan_fingerprints", "rows": int(fingerprint_bundle["fingerprints"].shape[0]), "cols": int(fingerprint_bundle["fingerprints"].shape[1])},
    ]
)
metadata_summary_df = pd.DataFrame(
    [
        {
            "tensor_artifacts_dir": str(tensor_artifacts_dir),
            "cell_line_metadata_arrow_path": str(cell_line_metadata_arrow_path),
            "n_examples": int(len(examples_df)),
            "n_unique_drugs": int(examples_df["drug"].nunique()),
            "n_unique_cell_lines": int(examples_df["cell_line"].nunique()),
            "n_unique_organs": int(examples_df["organ"].nunique()),
        }
    ]
)

print(
    f"Loaded {len(examples_df)} treatment examples with {examples_df['drug'].nunique()} drugs, {examples_df['cell_line'].nunique()} cell lines, and {examples_df['organ'].nunique()} organs."
)
display(tensor_summary_df)
display(metadata_summary_df)
display(examples_df.head())


/var/folders/rm/tp3kb8dj25dflt0f18q1f1_h0000gn/T/ipykernel_72717/2755787944.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dmso_bundle = torch.load(tensor_artifacts_dir

Loaded 26696 treatment examples with 377 drugs, 24 cell lines, and 10 organs.


/var/folders/rm/tp3kb8dj25dflt0f18q1f1_h0000gn/T/ipykernel_72717/2755787944.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  fingerprint_bundle = torch.load(tensor_artifa

,bundle,rows,cols
0,dmso_baselines,24,20061
1,treatment_expressions,26696,20061
2,morgan_fingerprints,377,2048


,tensor_artifacts_dir,cell_line_metadata_arrow_path,n_examples,n_unique_drugs,n_unique_cell_lines,n_unique_organs
0,/Users/aniruddh/Library/CloudStorage/OneDrive-...,/Users/aniruddh/.cache/huggingface/datasets/ve...,26696,377,24,10


,condition_key,cell_line,file_name,drug,concentration,concentration_unit,target_index,baseline_index,fingerprint_index,cell_name,organ
0,CVCL_0023|||(R)-Verapamil (hydrochloride)|||0....,CVCL_0023,CVCL_0023.h5ad,(R)-Verapamil (hydrochloride),0.05,uM,0,0,118,A549,Lung
1,CVCL_0023|||(R)-Verapamil (hydrochloride)|||0....,CVCL_0023,CVCL_0023.h5ad,(R)-Verapamil (hydrochloride),0.50,uM,1,0,118,A549,Lung
2,CVCL_0023|||(R)-Verapamil (hydrochloride)|||5|...,CVCL_0023,CVCL_0023.h5ad,(R)-Verapamil (hydrochloride),5.00,uM,2,0,118,A549,Lung
3,CVCL_0023|||(S)-Crizotinib|||0.05|||uM,CVCL_0023,CVCL_0023.h5ad,(S)-Crizotinib,0.05,uM,3,0,137,A549,Lung
4,CVCL_0023|||(S)-Crizotinib|||0.5|||uM,CVCL_0023,CVCL_0023.h5ad,(S)-Crizotinib,0.50,uM,4,0,137,A549,Lung


## Split Examples and Build DataLoaders


In [4]:
if SPLIT_MODE == "drug_blind":
    split_mode_note = "drug_blind: entire drugs are held out from train."
    split_unit_column = "drug"
    split_assignments = assign_group_blind_splits(
        examples_df,
        group_col="drug",
        split_fractions=SPLIT_FRACTIONS,
        seed=RANDOM_SEED,
    )
elif SPLIT_MODE == "tumor_blind":
    split_mode_note = "tumor_blind: this notebook implements cell-line-blind splits rather than Organ-level splits."
    split_unit_column = "cell_line"
    split_assignments = assign_group_blind_splits(
        examples_df,
        group_col="cell_line",
        split_fractions=SPLIT_FRACTIONS,
        seed=RANDOM_SEED,
    )
else:
    split_mode_note = "mixed: condition keys are held out, but every drug and cell line remains represented in train."
    split_unit_column = "condition_key"
    split_assignments = assign_mixed_split(
        examples_df,
        split_fractions=SPLIT_FRACTIONS,
        seed=RANDOM_SEED,
    )

split_examples_df = examples_df.copy()
split_examples_df["split"] = split_assignments.to_numpy()
validate_split_assignments(split_examples_df, SPLIT_MODE)

split_summary_df = build_split_summary(split_examples_df, SPLIT_FRACTIONS)
split_unit_summary_df = (
    split_examples_df.groupby("split")[split_unit_column]
    .nunique()
    .reindex(SPLIT_NAMES)
    .reset_index(name=f"unique_{split_unit_column}_count")
)
overlap_diagnostics_df = build_overlap_diagnostics(split_examples_df, SPLIT_MODE)

train_examples_df = split_examples_df.loc[split_examples_df["split"] == "train"].reset_index(drop=True)
val_examples_df = split_examples_df.loc[split_examples_df["split"] == "val"].reset_index(drop=True)
test_examples_df = split_examples_df.loc[split_examples_df["split"] == "test"].reset_index(drop=True)

train_dataset = TreatmentExampleDataset(train_examples_df, dmso_bundle, treatment_bundle, fingerprint_bundle)
val_dataset = TreatmentExampleDataset(val_examples_df, dmso_bundle, treatment_bundle, fingerprint_bundle)
test_dataset = TreatmentExampleDataset(test_examples_df, dmso_bundle, treatment_bundle, fingerprint_bundle)

train_generator = torch.Generator().manual_seed(RANDOM_SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    drop_last=False,
    generator=train_generator,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    drop_last=False,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    drop_last=False,
)

print(split_mode_note)
display(split_summary_df)
display(split_unit_summary_df)
display(overlap_diagnostics_df)


mixed: condition keys are held out, but every drug and cell line remains represented in train.


,split,target_rows,row_count,requested_fraction,realized_fraction,unique_drugs,unique_cell_lines,unique_organs
0,train,21356,21356,0.8,0.799970,377,24,10
1,val,2670,2670,0.1,0.100015,377,24,10
2,test,2670,2670,0.1,0.100015,377,24,10


,split,unique_condition_key_count
0,train,21356
1,val,2670
2,test,2670


,entity,pair,overlap_count
0,condition_key,train/val,0
1,condition_key,train/test,0
2,condition_key,val/test,0
3,drug,train/val,377
4,drug,train/test,377
5,drug,val/test,377
6,cell_line,train/val,24
7,cell_line,train/test,24
8,cell_line,val/test,24


## Inspect One Batch


In [5]:
batch_examples = {
    "train": next(iter(train_loader)),
    "val": next(iter(val_loader)),
    "test": next(iter(test_loader)),
}

batch_summary_rows = []
for split_name, batch in batch_examples.items():
    split_source_df = split_examples_df.loc[split_examples_df["split"] == split_name].set_index("condition_key")
    first_condition_key = batch["condition_key"][0]
    source_row = split_source_df.loc[first_condition_key]

    if batch["drug"][0] != source_row["drug"] or batch["cell_line"][0] != source_row["cell_line"]:
        raise ValueError("Batch metadata does not align with the split source table.")

    if batch["baseline_expression"].shape[0] > BATCH_SIZE:
        raise ValueError("A batch exceeded the configured batch size.")
    if batch["baseline_expression"].shape[1] != len(dmso_bundle["gene_ids"]):
        raise ValueError("Baseline expression width does not match the gene space.")
    if batch["drug_fingerprint"].shape[1] != fingerprint_bundle["fingerprints"].shape[1]:
        raise ValueError("Drug fingerprint width does not match the saved Morgan tensor.")
    if batch["target_expression"].shape[1] != len(treatment_bundle["gene_ids"]):
        raise ValueError("Target expression width does not match the gene space.")

    batch_summary_rows.append(
        {
            "split": split_name,
            "batch_rows": int(batch["baseline_expression"].shape[0]),
            "baseline_shape": tuple(batch["baseline_expression"].shape),
            "fingerprint_shape": tuple(batch["drug_fingerprint"].shape),
            "concentration_shape": tuple(batch["concentration"].shape),
            "target_shape": tuple(batch["target_expression"].shape),
            "first_condition_key": first_condition_key,
            "first_cell_line": batch["cell_line"][0],
            "first_drug": batch["drug"][0],
        }
    )

batch_summary_df = pd.DataFrame(batch_summary_rows)
display(batch_summary_df)


,split,batch_rows,baseline_shape,fingerprint_shape,concentration_shape,target_shape,first_condition_key,first_cell_line,first_drug
0,train,512,"(512, 20061)","(512, 2048)","(512,)","(512, 20061)",CVCL_0546|||Anastrozole|||5|||uM,CVCL_0546,Anastrozole
1,val,512,"(512, 20061)","(512, 2048)","(512,)","(512, 20061)",CVCL_0023|||18β-Glycyrrhetinic acid|||0.5|||uM,CVCL_0023,18β-Glycyrrhetinic acid
2,test,512,"(512, 20061)","(512, 2048)","(512,)","(512, 20061)",CVCL_0023|||(R)-Verapamil (hydrochloride)|||5|...,CVCL_0023,(R)-Verapamil (hydrochloride)
